<a href="https://colab.research.google.com/github/Murtuzasaifee/multi-model-rag/blob/master/Docling_Granite_Parsing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install docling[all]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 7.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.7/266.7 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 18.8 MB/s eta 0:00:00
   ━━━

In [ ]:
!docling --to html --to md --pipeline vlm --vlm-model granite_docling "https://arxiv.org/pdf/2501.17887"

2026-03-22 05:52:09.642827: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774158729.663286    2551 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774158729.670009    2551 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774158729.687144    2551 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774158729.687168    2551 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774158729.687172    2551 computation_placer.cc:177] computation placer alr

In [ ]:
# Prerequisites:
# pip install torch
# pip install docling_core
# pip install transformers

import torch
from docling_core.types.doc import DoclingDocument
from docling_core.types.doc.document import DocTagsDocument
from transformers import AutoProcessor, AutoModelForVision2Seq
from transformers.image_utils import load_image
from pathlib import Path

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load images
image = load_image("https://huggingface.co/ibm-granite/granite-docling-258M/resolve/main/assets/new_arxiv.png")

# Initialize processor and model
processor = AutoProcessor.from_pretrained("ibm-granite/granite-docling-258M")
model = AutoModelForVision2Seq.from_pretrained(
    "ibm-granite/granite-docling-258M",
    torch_dtype=torch.bfloat16,
    _attn_implementation="sdpa", # Changed to sdpa to avoid FlashAttention2 dependency
).to(DEVICE)

# Create input messages
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Convert this page to docling."}
        ]
    },
]

# Prepare inputs
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image], return_tensors="pt")
inputs = inputs.to(DEVICE)

# Generate outputs
generated_ids = model.generate(**inputs, max_new_tokens=8192)
prompt_length = inputs.input_ids.shape[1]
trimmed_generated_ids = generated_ids[:, prompt_length:]
doctags = processor.batch_decode(
    trimmed_generated_ids,
    skip_special_tokens=False,
)[0].lstrip()

print(f"DocTags: \n{doctags}\n")


# Populate document
doctags_doc = DocTagsDocument.from_doctags_and_image_pairs([doctags], [image])
# create a docling document
doc = DoclingDocument.load_from_doctags(doctags_doc, document_name="Document")
print(f"Markdown:\n{doc.export_to_markdown()}\n")

## export as any format.
# Path("out/").mkdir(parents=True, exist_ok=True)
# HTML:
# output_path_html = Path("out/") / "example.html"
# doc.save_as_html(output_path_html)
# Markdown:
# output_path_md = Path("out/") / "example.md"
# doc.save_as_markdown(output_path_md)


In [ ]:
# Prerequisites:
# pip install vllm
# pip install docling_core
# place page images you want to convert into "img/" dir

import time
import os
from vllm import LLM, SamplingParams
from transformers import AutoProcessor
from PIL import Image
from docling_core.types.doc import DoclingDocument
from docling_core.types.doc.document import DocTagsDocument
from pathlib import Path

# Configuration
MODEL_PATH = "ibm-granite/granite-docling-258M"
IMAGE_DIR = "img/"  # Place your page images here
OUTPUT_DIR = "out/"
PROMPT_TEXT = "Convert this page to docling."

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": PROMPT_TEXT},
        ],
    },
]


# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Initialize LLM
llm = LLM(model=MODEL_PATH, revision="untied", limit_mm_per_prompt={"image": 1})
processor = AutoProcessor.from_pretrained(MODEL_PATH)

sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=8192,
    skip_special_tokens=False,
)

# Load and prepare all images and prompts up front
batched_inputs = []
image_names = []

for img_file in sorted(os.listdir(IMAGE_DIR)):
    if img_file.lower().endswith((".png", ".jpg", ".jpeg")):
        img_path = os.path.join(IMAGE_DIR, img_file)
        with Image.open(img_path) as im:
            image = im.convert("RGB")

        prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
        batched_inputs.append({"prompt": prompt, "multi_modal_data": {"image": image}})
        image_names.append(os.path.splitext(img_file)[0])

# Run batch inference
start_time = time.time()
outputs = llm.generate(batched_inputs, sampling_params=sampling_params)

# Postprocess all results
for img_fn, output, input_data in zip(image_names, outputs, batched_inputs):
    doctags = output.outputs[0].text
    output_path_dt = Path(OUTPUT_DIR) / f"{img_fn}.dt"
    output_path_md = Path(OUTPUT_DIR) / f"{img_fn}.md"

    with open(output_path_dt, "w", encoding="utf-8") as f:
        f.write(doctags)

    # Convert to DoclingDocument and save markdown
    doctags_doc = DocTagsDocument.from_doctags_and_image_pairs([doctags], [input_data["multi_modal_data"]["image"]])
    doc = DoclingDocument.load_from_doctags(doctags_doc, document_name="Document")
    doc.save_as_markdown(output_path_md)

print(f"Total time: {time.time() - start_time:.2f} sec")

In [ ]:
from transformers import AutoProcessor, AutoModelForVision2Seq
from huggingface_hub import hf_hub_download
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model_path = "ibm-granite/granite-vision-3.3-2b"
processor = AutoProcessor.from_pretrained(model_path)
model = AutoModelForVision2Seq.from_pretrained(model_path).to(device)

# prepare image and text prompt, using the appropriate prompt template

img_path = hf_hub_download(repo_id=model_path, filename='/content/testimage.png')

conversation = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": img_path},
            {"type": "text", "text": "What is the version of MinerU?"},
        ],
    },
]
inputs = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(device)


# autoregressively complete prompt
output = model.generate(**inputs, max_new_tokens=100)
print(processor.decode(output[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

# ── Configure the standard pipeline ──────────────────────────────────────
pipeline_options = PdfPipelineOptions(
    do_layout=True,           # DocLayNet: layout segmentation
    do_table_structure=True,  # TableFormer: table parsing
    do_ocr=True,              # EasyOCR / Tesseract fallback for scanned pages
    generate_page_images=True,
    do_formula_enrichment=True,
    generate_picture_images=True,
)

# ── Build the converter ───────────────────────────────────────────────────
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

# ── Run on your arXiv PDF ─────────────────────────────────────────────────
source = "https://arxiv.org/pdf/2501.17887"
result = converter.convert(source)

# ── Export to Markdown ────────────────────────────────────────────────────
markdown_output = result.document.export_to_markdown()
with open("output.md", "w") as f:
    f.write(markdown_output)

# ── Export to HTML ────────────────────────────────────────────────────────
html_output = result.document.export_to_html()
with open("output.html", "w") as f:
    f.write(html_output)

print("✅ Done — DocLayNet + TableFormer pipeline")

[INFO] 2026-03-20 10:01:22,230 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-20 10:01:22,238 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-03-20 10:01:22,242 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.7.0/torch/PP-OCRv4/det/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-20 10:01:23,158 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2026-03-20 10:01:23,370 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-20 10:01:23,373 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-20 10:01:23,767 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-20 10:01:23,768 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-03-20 10:01:23,771 [RapidOCR] download_file.py:68: Initiat

✅ Done — DocLayNet + TableFormer pipeline


In [ ]:
!pip install docling_core

In [ ]:
import docling_core
import subprocess
result = subprocess.run(
    ["grep", "-r", "ImageRefMode", docling_core.__path__[0]],
    capture_output=True, text=True
)
print(result.stdout)

/usr/local/lib/python3.12/dist-packages/docling_core/transforms/chunker/hierarchical_chunker.py:from docling_core.types.doc.base import ImageRefMode
/usr/local/lib/python3.12/dist-packages/docling_core/transforms/chunker/hierarchical_chunker.py:        image_mode=ImageRefMode.PLACEHOLDER,
/usr/local/lib/python3.12/dist-packages/docling_core/transforms/serializer/latex.py:from docling_core.types.doc.base import ImageRefMode
/usr/local/lib/python3.12/dist-packages/docling_core/transforms/serializer/latex.py:    image_mode: ImageRefMode = ImageRefMode.PLACEHOLDER
/usr/local/lib/python3.12/dist-packages/docling_core/transforms/serializer/latex.py:        image_mode: ImageRefMode,
/usr/local/lib/python3.12/dist-packages/docling_core/transforms/serializer/latex.py:        if image_mode == ImageRefMode.PLACEHOLDER:
/usr/local/lib/python3.12/dist-packages/docling_core/transforms/serializer/latex.py:        elif image_mode == ImageRefMode.REFERENCED:
/usr/local/lib/python3.12/dist-packages/docl

## What each model actually does
```
Your PDF
    │
    ▼
┌─────────────────────────────────────────────────────┐
│  DocLayNet  (layout segmentation)                   │
│  → Classifies page regions:                         │
│    Text / Title / Table / Figure / List / Formula   │
└──────────────────────┬──────────────────────────────┘
                       │
          ┌────────────┴────────────┐
          ▼                         ▼
┌──────────────────┐     ┌─────────────────────────┐
│  OCR Engine      │     │  TableFormer             │
│  (EasyOCR /      │     │  → Reconstructs rows,    │
│   Tesseract)     │     │    columns, merged cells │
│  for scanned     │     │    → outputs HTML table  │
│  pages           │     └─────────────────────────┘
└──────────────────┘
          │
          ▼
┌─────────────────────────────────────────────────────┐
│  Structured DoclingDocument                         │
│  → .export_to_markdown()                            │
│  → .export_to_html()                                │
│  → .export_to_dict() / JSON                         │
└─────────────────────────────────────────────────────┘

In [2]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling_core.types.doc.base import ImageRefMode

pipeline_options = PdfPipelineOptions(
    do_layout=True,
    do_table_structure=True,
    do_ocr=False,
    do_formula_enrichment=True,
    generate_page_images=True,
    generate_picture_images=True,
    images_scale=2.0,
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

result = converter.convert("https://arxiv.org/pdf/2501.17887")

# ── HTML with embedded images ──────────────────────────────────────────────
html_output = result.document.export_to_html(
    image_mode=ImageRefMode.EMBEDDED
)
with open("output_with_images.html", "w") as f:
    f.write(html_output)

# ── Markdown with embedded images ──────────────────────────────────────────
md_output = result.document.export_to_markdown(
    image_mode=ImageRefMode.EMBEDDED
)
with open("output_with_images.md", "w") as f:
    f.write(md_output)

print("✅ Done")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:86: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


✅ Done


Key difference vs your current setup

VLM mode (what you had)| Standard Pipeline (recommended)Layout detectionGranite-Docling 258M (one-shot)DocLayNet (dedicated model)Table parsingInferred by VLMTableFormer (specialized)Formula handlingWeakBetter with do_formula_enrichment=TrueSpeedFasterSlower but far more accurateGPU neededYesNo — CPU works fine

